# American Time Use Survey (ATUS) — Data Extraction

**Purpose**: Extract time-use data for married men and women, broken out by whether they have children, focusing on **household production** — the full set of activities a couple must do to maintain a household.

**Key question**: Do the news stories claiming "men don't share the housework" only count *housework* (cleaning, cooking), or do they include everything men actually contribute — market labor, commute, childcare, home maintenance, volunteering, coaching kids' teams, driving to activities, and more?

## What is Household Production?

We define household production broadly as **everything a couple is reasonably required to do to maintain a household**, including:

| Category | Examples |
|----------|----------|
| **Housework & home maintenance** | Interior cleaning, laundry, cooking, food prep/cleanup, interior repairs, exterior repairs, **lawn care, landscaping, sprinkler repair**, vehicle repair, appliance repair, household management |
| **Childcare (household)** | Physical care, playing, reading, homework help, attending kids' events, **driving kids to/from school and activities**, medical care |
| **Care for non-household members** | Caring for nonhh children, helping nonhh adults |
| **Market labor** | Work at main job, other jobs, work-related activities, income-generating hobbies |
| **Commute** | Travel to/from work |
| **Shopping** | Groceries, gas, food, other consumer purchases |
| **Professional & personal services** | Banking, medical services, childcare services |
| **Household services** | Using repair, maintenance, lawn/garden, vehicle services |
| **Civic & government** | Civic obligations, government services |
| **Volunteering** | Coaching, mentoring, building, food prep, public health/safety, admin |
| **Travel for all the above** | Travel to shopping, childcare, volunteering, civic activities |

### Why include commute and market labor?

A couple cannot maintain a household without income. The hours spent earning that income — including the commute — are hours *not available* for other household production. If one partner works 40 hours/week plus 5 hours commuting, those 45 hours are their contribution to household production, just as the other partner's 15 hours of cooking + 20 hours of childcare are theirs.

### Why include volunteering and community work?

Coaching a kid's little league team, helping at after-school programs, organizing community events — these are often things parents (especially fathers) do as part of maintaining a household with children. They substitute for paid services and are required labor for the household.

---

**Data source**: ATUS 2018 (American Time Use Survey, Bureau of Labor Statistics).  
Downloaded from a GitHub mirror (`changyaochen/atus-2018`) because BLS blocks automated downloads.  
Original data: https://www.bls.gov/tus/datafiles.htm  
Activity coding lexicon: https://www.bls.gov/tus/lexiconwex2018.pdf

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import re
import os
from collections import defaultdict

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 150)

RAW_DIR = 'data/raw'
OUT_DIR = 'data'
os.makedirs(OUT_DIR, exist_ok=True)

## 2. Download ATUS Data

The BLS website blocks automated downloads (Akamai bot detection). We download from a GitHub mirror that hosts the same files.

In [2]:
import urllib.request

BASE_URL = 'https://raw.githubusercontent.com/changyaochen/atus-2018/master/data'

FILES = {
    'atussum_2018.dat':  f'{BASE_URL}/atussum-2018/atussum_2018.dat',
    'atussum_2018.do':   f'{BASE_URL}/atussum-2018/atussum_2018.do',
    'Sum2018_info.txt':  f'{BASE_URL}/atussum-2018/Sum2018_info.txt',
    'atusresp_2018.dat': f'{BASE_URL}/atusresp-2018/atusresp_2018.dat',
    'atusresp_2018.do':  f'{BASE_URL}/atusresp-2018/atusresp_2018.do',
    'atusact_2018.dat':  f'{BASE_URL}/atusact-2018/atusact_2018.dat',
    'atusact_2018.do':   f'{BASE_URL}/atusact-2018/atusact_2018.do',
    'atuswho_2018.dat':  f'{BASE_URL}/atuswho-2018/atuswho_2018.dat',
    'atuswho_2018.do':   f'{BASE_URL}/atuswho-2018/atuswho_2018.do',
}

os.makedirs(RAW_DIR, exist_ok=True)

for filename, url in FILES.items():
    filepath = os.path.join(RAW_DIR, filename)
    if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
        print(f'  Already exists: {filename} ({os.path.getsize(filepath):,} bytes)')
    else:
        print(f'  Downloading: {filename} ...')
        urllib.request.urlretrieve(url, filepath)
        print(f'    Saved: {filepath} ({os.path.getsize(filepath):,} bytes)')

print('\nAll files ready.')

  Already exists: atussum_2018.dat (8,706,685 bytes)
  Already exists: atussum_2018.do (35,582 bytes)
  Already exists: Sum2018_info.txt (2,641 bytes)
  Already exists: atusresp_2018.dat (4,896,623 bytes)
  Already exists: atusresp_2018.do (65,613 bytes)
  Already exists: atusact_2018.dat (20,964,037 bytes)
  Already exists: atusact_2018.do (6,079 bytes)
  Already exists: atuswho_2018.dat (6,013,506 bytes)
  Already exists: atuswho_2018.do (1,507 bytes)

All files ready.


## 3. Parse Column Schema from Stata .do File

The ATUS summary file (`atussum`) is a comma-separated text file with 426 columns. The Stata `.do` file contains column names and activity labels.

In [3]:
# Parse column names from the Stata import block
with open(os.path.join(RAW_DIR, 'atussum_2018.do'), 'r', errors='replace') as f:
    do_text = f.read()

m = re.search(r'import delimited\s+(.*?)using', do_text, re.DOTALL)
all_cols = [c.strip() for c in m.group(1).strip().split('\n') if c.strip()]
print(f'Total columns: {len(all_cols)}')

# Parse activity labels
activity_labels = dict(re.findall(r'label variable (t\d{6})\s+"(.*?)"', do_text))
print(f'Activity labels: {len(activity_labels)}')

# Identify activity columns (t-prefixed, 6-digit code)
act_cols = [c for c in all_cols if c[0] == 't' and len(c) == 7 and c[1].isdigit()]
demo_cols = [c for c in all_cols if c not in act_cols]
print(f'Demographic/other columns: {len(demo_cols)}')
print(f'Activity columns: {len(act_cols)}')

Total columns: 426
Activity labels: 402
Demographic/other columns: 24
Activity columns: 402


## 4. Load ATUS Summary Data

The summary file contains one row per respondent with total minutes spent on each 6-digit activity during the diary day.

In [4]:
df = pd.read_csv(os.path.join(RAW_DIR, 'atussum_2018.dat'), sep=',',
                 names=all_cols, skiprows=1, low_memory=False)

# Convert all columns except tucaseid to numeric
for c in all_cols:
    if c != 'tucaseid':
        df[c] = pd.to_numeric(df[c], errors='coerce')

print(f'Loaded {len(df):,} respondents x {len(df.columns)} variables')
print(f'\nKey demographics:')
print(f'  Sex (tesex): 1=Male, 2=Female')
print(f'  {df["tesex"].value_counts().sort_index().to_dict()}')
print(f'\n  Spouse present (trsppres): 1=Spouse, 2=Unmarried partner, 3=Neither')
print(f'  {df["trsppres"].value_counts().sort_index().to_dict()}')
print(f'\n  Age range: {df["teage"].min():.0f}-{df["teage"].max():.0f} (mean={df["teage"].mean():.1f})')

Loaded 9,593 respondents x 426 variables

Key demographics:
  Sex (tesex): 1=Male, 2=Female
  {1: 4318, 2: 5275}

  Spouse present (trsppres): 1=Spouse, 2=Unmarried partner, 3=Neither
  {1: 4571, 2: 361, 3: 4661}

  Age range: 15-85 (mean=50.6)


## 5. Filter: Married Respondents

We keep only respondents with a spouse present in the household (`trsppres == 1`).

In [5]:
married = df[df['trsppres'] == 1].copy()
print(f'Married respondents: {len(married):,}')
print(f'  Men:   {(married["tesex"]==1).sum():,}')
print(f'  Women: {(married["tesex"]==2).sum():,}')

# Create has_kids flag: any household children under 18
married['has_kids'] = (married['trchildnum'] > 0).astype(int)

print(f'\n  With kids:    {married["has_kids"].sum():,}')
print(f'  Without kids:  {(married["has_kids"]==0).sum():,}')

print(f'\n--- Four-group breakdown ---')
for sex in [1, 2]:
    for kids in [0, 1]:
        n = ((married['tesex']==sex) & (married['has_kids']==kids)).sum()
        label = f"{'Men' if sex==1 else 'Women'} {'w/ kids' if kids==1 else 'no kids'}"
        print(f'  {label:20s}: {n:,}')

Married respondents: 4,571
  Men:   2,253
  Women: 2,318

  With kids:    2,371
  Without kids:  2,200

--- Four-group breakdown ---
  Men no kids         : 1,083
  Men w/ kids         : 1,170
  Women no kids       : 1,117
  Women w/ kids       : 1,201


## 6. Define Household Production Categories

We map ATUS activity codes to broad categories of household production. Each category includes both the activity itself and travel related to that activity.

### ATUS Tier 1 Categories

| Code | Category |
|------|----------|
| 01 | Personal care (sleeping, grooming) — *not included* |
| 02 | Household activities (cleaning, cooking, maintenance, lawn, pets) |
| 03 | Caring for household members (children, adults) |
| 04 | Caring for non-household members (children, adults) |
| 05 | Work & work-related activities |
| 06 | Education |
| 07 | Consumer purchases (shopping) |
| 08 | Professional & personal services |
| 09 | Household services (using paid help) |
| 10 | Government services & civic obligations |
| 11 | Eating & drinking — *not included* |
| 12 | Socializing & leisure — *not included* |
| 13 | Sports & recreation — *not included* |
| 14 | Religious activities — *not included* |
| 15 | Volunteer activities |
| 16 | Telephone calls — *not included* |
| 18 | Travel (categorized by purpose) |


In [6]:
def get_tier1_cols(prefix, cols=act_cols):
    '''Get all activity columns matching a tier1 prefix (e.g., '02' -> all t02xxxx)'''
    return [c for c in cols if c[1:3] == prefix]

def get_specific_cols(codes, cols=act_cols):
    '''Get specific columns by 6-digit code (without 't' prefix)'''
    return [f't{code}' for code in codes if f't{code}' in cols]

# -- Primary activity categories --

categories = {
    'housework_home_maintenance': {
        'description': (
            'All household activities: interior cleaning, laundry, sewing, food prep & cleanup, '
            'interior arrangement/repairs, EXTERIOR cleaning/repairs/decoration, '
            'LAWN & GARDEN CARE (mowing, landscaping, sprinklers), ponds/pools, '
            'pet care, vehicle repair/maintenance, appliance/tool repair, household management'
        ),
        'tier1': '02',
        'cols': get_tier1_cols('02'),
    },
    'childcare_household': {
        'description': (
            'Caring for household children: physical care, reading, playing, arts & crafts, '
            'sports, talking, organizing, looking after, ATTENDING KIDS EVENTS, waiting, '
            'PICKING UP/DROPPING OFF, homework help, school conferences, homeschooling, '
            'medical care. Also includes caring for household adults.'
        ),
        'tier1': '03',
        'cols': get_tier1_cols('03'),
    },
    'care_nonhh_members': {
        'description': (
            'Caring for non-household members: nonhh children (coaching little league, '
            'helping with homework, attending events, driving), nonhh adults '
            '(housework help, maintenance help, financial help)'
        ),
        'tier1': '04',
        'cols': get_tier1_cols('04'),
    },
    'market_labor': {
        'description': (
            'Market labor: work at main job, other jobs, security procedures at work, '
            'waiting for work, socializing/eating as part of job, income-generating hobbies '
            '(crafts, performances, services, rental property), job search & interviewing'
        ),
        'tier1': '05',
        'cols': get_tier1_cols('05'),
    },
    'shopping': {
        'description': (
            'Consumer purchases: grocery shopping, purchasing gas, purchasing food (not groceries), '
            'general shopping, comparison shopping'
        ),
        'tier1': '07',
        'cols': get_tier1_cols('07'),
    },
    'prof_personal_services': {
        'description': (
            'Professional & personal services: paid childcare, banking, financial services, '
            'legal services, medical/health services, personal care, real estate, veterinary'
        ),
        'tier1': '08',
        'cols': get_tier1_cols('08'),
    },
    'household_services': {
        'description': (
            'Using household services: interior cleaning, meal prep, clothing repair, '
            'home maintenance/repair/decor/construction, pet services, lawn & garden services, '
            'vehicle maintenance/repair'
        ),
        'tier1': '09',
        'cols': get_tier1_cols('09'),
    },
    'civic_gov': {
        'description': (
            'Government services & civic obligations: police/fire, social services, '
            'licenses/fines/fees/taxes, civic obligations & participation'
        ),
        'tier1': '10',
        'cols': get_tier1_cols('10'),
    },
    'volunteer': {
        'description': (
            'Volunteer activities: computer use, organizing, reading, phone calls, writing, '
            'fundraising, food prep, collecting/delivering goods, providing care, '
            'TEACHING/LEADING/COUNSELING/MENTORING (coaching), building houses/structures, '
            'indoor/outdoor maintenance/repair/cleanup, performing, serving at events, '
            'attending meetings, public health & safety activities'
        ),
        'tier1': '15',
        'cols': get_tier1_cols('15'),
    },
}

# -- Travel categories (subsets of tier 1 = 18) --

travel_categories = {
    'commute': {
        'description': 'Commute: travel related to working, work-related activities, income-generating activities, job search',
        'cols': get_specific_cols(['180501', '180502', '180503', '180504']),
    },
    'travel_housework': {
        'description': 'Travel related to household activities (getting supplies, taking pets to vet, etc.)',
        'cols': get_specific_cols([
            '180201', '180202', '180203', '180204', '180205',
            '180206', '180207', '180208', '180209'
        ]),
    },
    'travel_childcare': {
        'description': 'Travel related to childcare (driving kids to school, activities, medical appointments)',
        'cols': get_specific_cols(['180301', '180302', '180303', '180304', '180305']),
    },
    'travel_nonhh_care': {
        'description': 'Travel related to caring for non-household members',
        'cols': get_specific_cols(['180401', '180402', '180403', '180404', '180405']),
    },
    'travel_shopping': {
        'description': 'Travel related to shopping (groceries, gas, other purchases)',
        'cols': get_specific_cols(['180701', '180702', '180703', '180704']),
    },
    'travel_volunteer': {
        'description': 'Travel related to volunteering',
        'cols': get_specific_cols(['181501']),
    },
    'travel_civic': {
        'description': 'Travel related to government services & civic obligations',
        'cols': get_specific_cols(['181001', '181002']),
    },
}

all_categories = {**categories, **travel_categories}

print('=' * 80)
print('HOUSEHOLD PRODUCTION CATEGORIES')
print('=' * 80)
for name, info in all_categories.items():
    print(f'\n{name} ({len(info["cols"])} codes):')
    print(f'  {info["description"][:120]}...')
    for c in info['cols'][:4]:
        print(f'    {c}: {activity_labels.get(c, "?")}')
    if len(info['cols']) > 4:
        print(f'    ... and {len(info["cols"]) - 4} more')

HOUSEHOLD PRODUCTION CATEGORIES

housework_home_maintenance (32 codes):
  All household activities: interior cleaning, laundry, sewing, food prep & cleanup, interior arrangement/repairs, EXTERIO...
    t020101: Interior cleaning
    t020102: Laundry
    t020103: Sewing, repairing, and maintaining textiles
    t020104: Storing interior hh items, inc. food
    ... and 28 more

childcare_household (33 codes):
  Caring for household children: physical care, reading, playing, arts & crafts, sports, talking, organizing, looking afte...
    t030101: Physical care for hh children
    t030102: Reading to/with hh children
    t030103: Playing with hh children, not sports
    t030104: Arts and crafts with hh children
    ... and 29 more

care_nonhh_members (36 codes):
  Caring for non-household members: nonhh children (coaching little league, helping with homework, attending events, drivi...
    t040101: Physical care for nonhh children
    t040102: Reading to/with nonhh children
    t040103: Pla

## 7. Compute Category Time Variables

For each respondent, sum the minutes across all activity codes within each category.

In [7]:
for name, info in all_categories.items():
    married[name] = married[info['cols']].sum(axis=1)

# Total household production = sum of all categories
category_cols = list(all_categories.keys())
married['total_household_production'] = married[category_cols].sum(axis=1)

# Create group label for convenience
married['sex_label'] = married['tesex'].map({1: 'Men', 2: 'Women'})
married['kids_label'] = married['has_kids'].map({0: 'no kids', 1: 'w/ kids'})
married['group'] = married['sex_label'] + ' ' + married['kids_label']

print('Category variables computed:')
for c in category_cols + ['total_household_production']:
    print(f'  {c}: mean={married[c].mean():.1f} min/day, max={married[c].max():.0f} min/day')

Category variables computed:
  housework_home_maintenance: mean=132.7 min/day, max=985 min/day
  childcare_household: mean=48.8 min/day, max=910 min/day
  care_nonhh_members: mean=10.8 min/day, max=900 min/day
  market_labor: mean=167.6 min/day, max=1330 min/day
  shopping: mean=25.2 min/day, max=770 min/day
  prof_personal_services: mean=4.1 min/day, max=405 min/day
  household_services: mean=1.0 min/day, max=540 min/day
  civic_gov: mean=0.4 min/day, max=353 min/day
  volunteer: mean=10.2 min/day, max=620 min/day
  commute: mean=14.1 min/day, max=555 min/day
  travel_housework: mean=2.8 min/day, max=755 min/day
  travel_childcare: mean=7.1 min/day, max=225 min/day
  travel_nonhh_care: mean=2.6 min/day, max=313 min/day
  travel_shopping: mean=15.8 min/day, max=485 min/day
  travel_volunteer: mean=1.3 min/day, max=210 min/day
  travel_civic: mean=0.1 min/day, max=90 min/day
  total_household_production: mean=444.6 min/day, max=1390 min/day


## 8. Quick Summary: Weighted Mean Minutes per Day

ATUS uses survey weights (`tufinlwgt`). We compute weighted means for each group.

In [8]:
def weighted_mean(data, values, weights):
    return (data[values] * data[weights]).sum() / data[weights].sum()

groups = ['Men w/ kids', 'Men no kids', 'Women w/ kids', 'Women no kids']
display_cols = category_cols + ['total_household_production']

summary = pd.DataFrame(index=groups)
for grp in groups:
    subset = married[married['group'] == grp]
    for col in display_cols:
        summary.loc[grp, col] = weighted_mean(subset, col, 'tufinlwgt')

# Convert to hours per week (x 7 / 60)
summary_weekly = summary * 7 / 60

print('Mean minutes per day (weighted):')
print(summary.round(1).to_string())
print('\n\nMean hours per week (weighted):')
print(summary_weekly.round(1).to_string())

Mean minutes per day (weighted):
               housework_home_maintenance  childcare_household  care_nonhh_members  market_labor  shopping  prof_personal_services  household_services  civic_gov  volunteer  commute  travel_housework  travel_childcare  travel_nonhh_care  travel_shopping  travel_volunteer  travel_civic  total_household_production
Men w/ kids                          84.3                 53.2                 4.0         328.5      17.7                     2.2                 1.5        0.8        7.4     30.4               2.3               9.5                1.1             13.7               0.9           0.2                       557.7
Men no kids                         108.7                  4.6                14.8         200.3      16.5                     7.7                 2.2        0.6        8.9     19.9               2.8               1.9                3.3             13.5               1.3           0.1                       407.2
Women w/ kids            

## 9. Save Extracted Dataset

Save a clean CSV with demographics + computed category variables for use in the analysis notebook.

In [9]:
keep_cols = [
    'tucaseid', 'tufinlwgt', 'teage', 'tesex', 'trsppres', 'trchildnum',
    'tryhhchild', 'telfs', 'trdpftpt', 'tehruslt', 'tudiaryday', 'trholiday',
    'peeduca', 'ptdtrace', 'pehspnon', 'gtmetsta', 'tespempnot', 'trspftpt',
    'has_kids', 'sex_label', 'kids_label', 'group',
] + category_cols + ['total_household_production']

output_df = married[keep_cols].copy()
output_path = os.path.join(OUT_DIR, 'atus_extracted.csv')
output_df.to_csv(output_path, index=False)

print(f'Saved {len(output_df):,} rows x {len(output_df.columns)} columns to {output_path}')
print(f'\nColumns:')
for c in output_df.columns:
    print(f'  {c}')

Saved 4,571 rows x 39 columns to data/atus_extracted.csv

Columns:
  tucaseid
  tufinlwgt
  teage
  tesex
  trsppres
  trchildnum
  tryhhchild
  telfs
  trdpftpt
  tehruslt
  tudiaryday
  trholiday
  peeduca
  ptdtrace
  pehspnon
  gtmetsta
  tespempnot
  trspftpt
  has_kids
  sex_label
  kids_label
  group
  housework_home_maintenance
  childcare_household
  care_nonhh_members
  market_labor
  shopping
  prof_personal_services
  household_services
  civic_gov
  volunteer
  commute
  travel_housework
  travel_childcare
  travel_nonhh_care
  travel_shopping
  travel_volunteer
  travel_civic
  total_household_production


## 10. Verification: Activity Codes in Each Category

For transparency and reproducibility, print every ATUS activity code included in each category.

In [10]:
for name, info in all_categories.items():
    print()
    print('=' * 80)
    print(f'{name}: {info["description"]}')
    print('=' * 80)
    for c in info['cols']:
        code = c[1:]
        tier1, tier2, tier3 = code[:2], code[2:4], code[4:6]
        label = activity_labels.get(c, '?')
        print(f'  {c} ({tier1}.{tier2}.{tier3}): {label}')


housework_home_maintenance: All household activities: interior cleaning, laundry, sewing, food prep & cleanup, interior arrangement/repairs, EXTERIOR cleaning/repairs/decoration, LAWN & GARDEN CARE (mowing, landscaping, sprinklers), ponds/pools, pet care, vehicle repair/maintenance, appliance/tool repair, household management
  t020101 (02.01.01): Interior cleaning
  t020102 (02.01.02): Laundry
  t020103 (02.01.03): Sewing, repairing, and maintaining textiles
  t020104 (02.01.04): Storing interior hh items, inc. food
  t020199 (02.01.99): Housework, n.e.c.*
  t020201 (02.02.01): Food and drink preparation
  t020202 (02.02.02): Food presentation
  t020203 (02.02.03): Kitchen and food clean-up
  t020299 (02.02.99): Food and drink prep, presentation, and clean-up, n.e.c.*
  t020301 (02.03.01): Interior arrangement, decoration, and repairs
  t020302 (02.03.02): Building and repairing furniture
  t020303 (02.03.03): Heating and cooling
  t020399 (02.03.99): Interior maintenance, repair, an

  t090301 (09.03.01): Using pet services
  t090399 (09.03.99): Using pet services, n.e.c.*
  t090401 (09.04.01): Using lawn and garden services
  t090501 (09.05.01): Using vehicle maintenance or repair services
  t090502 (09.05.02): Waiting associated with vehicle main. or repair svcs
  t090599 (09.05.99): Using vehicle maint. and repair svcs, n.e.c.*
  t099999 (09.99.99): Using household services, n.e.c.*

civic_gov: Government services & civic obligations: police/fire, social services, licenses/fines/fees/taxes, civic obligations & participation
  t100101 (10.01.01): Using police and fire services
  t100102 (10.01.02): Using social services
  t100103 (10.01.03): Obtaining licenses and paying fines, fees, taxes
  t100199 (10.01.99): Using government services, n.e.c.*
  t100201 (10.02.01): Civic obligations and participation
  t100299 (10.02.99): Civic obligations and participation, n.e.c.*
  t100304 (10.03.04): Waiting associated with using government services
  t100305 (10.03.05): Wa

---

## Summary

This notebook extracted ATUS 2018 data for **4,571 married respondents** and computed time spent on **16 categories of household production** (9 activity categories + 7 travel categories), plus a total.

The four groups for analysis:
- Married men with kids (n=1,170)
- Married men without kids (n=1,083)
- Married women with kids (n=1,201)
- Married women without kids (n=1,117)

**Key preliminary finding**: When you count *all* household production — not just housework but market labor, commute, childcare, shopping, volunteering, and all related travel — married men and women with kids spend nearly identical total time: **~65 hours/week** each. The difference is in the *composition*, not the total.

→ See `02_analysis.ipynb` for the full analysis and visualizations.